In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader

NVIDIA A100-SXM4-80GB


In [2]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path
from google.colab import drive
import cv2
import numpy as np
from tqdm import tqdm

In [3]:
#First mount Drive for the dataset if it is not mounted or session is restarted.
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Clone or pull the latest code
import os
REPO = "LaneDetection"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/abdullahtapanci/LaneDetection.git /content/{REPO}
else:
    !cd /content/{REPO} && git pull

%cd /content/{REPO}

Cloning into '/content/LaneDetection'...
remote: Enumerating objects: 409, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 409 (delta 21), reused 32 (delta 9), pack-reused 362 (from 1)
Receiving objects: 100% (409/409), 34.57 MiB | 12.57 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/LaneDetection


EXTRACT THE DATA INTO COLAB ENVIRONMENT
1.SETUP

In [5]:
# =========================
# CULane local extraction setup (Colab)
# =========================

# 1) Configure paths
DRIVE_CULANE_DIR = Path("/content/drive/MyDrive/CULane_Dataset")
LOCAL_WORK_DIR   = Path("/content")
LOCAL_ARCHIVE_DIR = LOCAL_WORK_DIR / "culane_archives"
LOCAL_CULANE_DIR  = LOCAL_WORK_DIR / "CULane"

LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

# 2) Which archives to process
# Include only files that exist in your Drive folder
ARCHIVES = [
    "driver_23_30frame.tar.gz",
    "driver_100_30frame.tar.gz",
    "driver_161_90frame.tar.gz",
    "driver_182_30frame.tar.gz",
    "driver_193_90frame.tar.gz",
    "driver_37_30frame.tar.gz",
    "annotations_new.tar.gz",
    "laneseg_label_w16.tar.gz",
    "laneseg_label_w16_test.zip",
    "list.tar.gz",
]


# 3) Copy from Drive -> local (skip if already copied with same size)
def copy_if_needed(src: Path, dst: Path):
    if not src.exists():
        print(f"[WARN] Missing on Drive: {src.name}")
        return False
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f"[SKIP] Already copied: {src.name}")
        return True
    print(f"[COPY] {src.name}")
    shutil.copy2(src, dst)
    return True

copied = []
for name in ARCHIVES:
    src = DRIVE_CULANE_DIR / name
    dst = LOCAL_ARCHIVE_DIR / name
    ok = copy_if_needed(src, dst)
    if ok:
        copied.append(dst)

print(f"\nTotal ready archives: {len(copied)}")

[COPY] driver_23_30frame.tar.gz
[COPY] driver_100_30frame.tar.gz
[COPY] driver_161_90frame.tar.gz
[COPY] driver_182_30frame.tar.gz
[COPY] driver_193_90frame.tar.gz
[COPY] driver_37_30frame.tar.gz
[COPY] annotations_new.tar.gz
[COPY] laneseg_label_w16.tar.gz
[COPY] laneseg_label_w16_test.zip
[COPY] list.tar.gz

Total ready archives: 10


EXTRACTION PROCESS

In [6]:
# =========================
# Extract archives into /content/CULane
# =========================
from pathlib import Path
import tarfile, zipfile

LOCAL_ARCHIVE_DIR = Path("/content/culane_archives")
LOCAL_CULANE_DIR  = Path("/content/CULane")
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

def extract_tar_gz(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT tar] {archive_path.name}")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=out_dir)

def extract_zip(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT zip] {archive_path.name}")
    with zipfile.ZipFile(archive_path, "r") as z:
        z.extractall(path=out_dir)

for p in sorted(LOCAL_ARCHIVE_DIR.iterdir()):
    if p.suffixes[-2:] == [".tar", ".gz"]:
        extract_tar_gz(p, LOCAL_CULANE_DIR)
    elif p.suffix == ".zip":
        extract_zip(p, LOCAL_CULANE_DIR)
    else:
        print(f"[SKIP] Unknown format: {p.name}")

print("\nExtraction complete.")

[EXTRACT tar] annotations_new.tar.gz


/tmp/ipykernel_2596/3457412522.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=out_dir)


[EXTRACT tar] driver_100_30frame.tar.gz
[EXTRACT tar] driver_161_90frame.tar.gz
[EXTRACT tar] driver_182_30frame.tar.gz
[EXTRACT tar] driver_193_90frame.tar.gz
[EXTRACT tar] driver_23_30frame.tar.gz
[EXTRACT tar] driver_37_30frame.tar.gz
[EXTRACT tar] laneseg_label_w16.tar.gz
[EXTRACT zip] laneseg_label_w16_test.zip
[EXTRACT tar] list.tar.gz

Extraction complete.


VERIFY

In [7]:
# =========================
# Quick verification
# =========================
from pathlib import Path

root = Path("/content/CULane")

# count common file types
jpg_count = len(list(root.rglob("*.jpg")))
png_count = len(list(root.rglob("*.png")))
lines_count = len(list(root.rglob("*.lines.txt")))
list_files = list(root.rglob("*list*")) + list(root.rglob("train*.txt")) + list(root.rglob("val*.txt")) + list(root.rglob("test*.txt"))

print(f"Root: {root}")
print(f"JPG count      : {jpg_count}")
print(f"PNG count      : {png_count}")
print(f"lines.txt count: {lines_count}")
print(f"List-like files: {len(list_files)}")

# Show some list files for sanity
for f in sorted(set(list_files))[:20]:
    print(" -", f)

Root: /content/CULane
JPG count      : 133235
PNG count      : 135014
lines.txt count: 134692
List-like files: 15
 - /content/CULane/list
 - /content/CULane/list/test.txt
 - /content/CULane/list/test_split/test0_normal.txt
 - /content/CULane/list/test_split/test1_crowd.txt
 - /content/CULane/list/test_split/test2_hlight.txt
 - /content/CULane/list/test_split/test3_shadow.txt
 - /content/CULane/list/test_split/test4_noline.txt
 - /content/CULane/list/test_split/test5_arrow.txt
 - /content/CULane/list/test_split/test6_curve.txt
 - /content/CULane/list/test_split/test7_cross.txt
 - /content/CULane/list/test_split/test8_night.txt
 - /content/CULane/list/train.txt
 - /content/CULane/list/train_gt.txt
 - /content/CULane/list/val.txt
 - /content/CULane/list/val_gt.txt


NOW, WE HAVE TO CREATE BINARY MASK FOLDER AND ALSO CONSTRUCT MANIFEST TXT FILES FOR TRAINING AND VALIDATION

In [8]:
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
from tqdm import tqdm

CULANE_ROOT = Path("/content/CULane")

TRAIN_GT = CULANE_ROOT / "list" / "train_gt.txt"
VAL_GT   = CULANE_ROOT / "list" / "val_gt.txt"

TRAIN_MANIFEST = CULANE_ROOT / "train.txt"
VAL_MANIFEST   = CULANE_ROOT / "val.txt"

BINARY_MASK_DIR = CULANE_ROOT / "binary_label"
BINARY_MASK_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
def clean_rel_path(path_text):
    return str(path_text).strip().lstrip("/")

def resolve_path(root, rel_path):
    return root / clean_rel_path(rel_path)

def get_driver_name(rel_path):
    parts = Path(clean_rel_path(rel_path)).parts
    return parts[0] if parts else "unknown"

def binary_mask_rel_from_seg_rel(seg_rel):
    seg_rel = clean_rel_path(seg_rel)
    parts = Path(seg_rel).parts

    if parts[0] == "laneseg_label_w16":
        return str(Path("binary_label", *parts[1:]))

    return str(Path("binary_label") / Path(seg_rel).name)

def create_binary_mask(seg_path, binary_path):
    seg = cv2.imread(str(seg_path), cv2.IMREAD_GRAYSCALE)

    if seg is None:
        raise FileNotFoundError(f"Could not read segmentation mask: {seg_path}")

    binary = (seg > 0).astype(np.uint8) * 255

    binary_path.parent.mkdir(parents=True, exist_ok=True)

    ok = cv2.imwrite(str(binary_path), binary)
    if not ok:
        raise RuntimeError(f"Could not write binary mask: {binary_path}")


In [10]:
def build_lanedataset_manifest(
    gt_file,
    out_manifest,
    root,
    overwrite_binary=False,
    skip_missing=True,
):
    """
    Builds manifest for your current LaneDataset format:

        image_path binary_mask_path instance_mask_path

    CULane gt format:

        image_path segmentation_mask_path lane1 lane2 lane3 lane4

    If skip_missing=True, rows for missing drivers/files are skipped.
    This is useful because your notebook skipped driver_23_30frame.
    """

    assert gt_file.exists(), f"Missing gt file: {gt_file}"

    lines = [
        line.strip()
        for line in gt_file.read_text().splitlines()
        if line.strip()
    ]

    manifest_rows = []
    skipped = []
    bad_rows = []

    kept_by_driver = Counter()
    skipped_by_driver = Counter()

    for line_idx, line in enumerate(tqdm(lines, desc=f"Building {out_manifest.name}"), start=1):
        parts = line.split()

        if len(parts) < 2:
            bad_rows.append((line_idx, line))
            continue

        img_rel = clean_rel_path(parts[0])
        seg_rel = clean_rel_path(parts[1])

        img_path = resolve_path(root, img_rel)
        seg_path = resolve_path(root, seg_rel)

        binary_rel = binary_mask_rel_from_seg_rel(seg_rel)
        binary_path = resolve_path(root, binary_rel)

        driver = get_driver_name(img_rel)

        missing_reasons = []

        if not img_path.exists():
            missing_reasons.append(("image", img_path))

        if not seg_path.exists():
            missing_reasons.append(("segmentation", seg_path))

        if missing_reasons:
            skipped_by_driver[driver] += 1
            skipped.append((line_idx, driver, missing_reasons))

            if skip_missing:
                continue

            print("Missing files:")
            for kind, path in missing_reasons:
                print(f"[{kind}] line {line_idx}: {path}")

            raise FileNotFoundError(f"Missing files at line {line_idx}")

        if overwrite_binary or not binary_path.exists():
            create_binary_mask(seg_path, binary_path)

        manifest_rows.append(f"{img_rel} {binary_rel} {seg_rel}")
        kept_by_driver[driver] += 1

    if bad_rows:
        print("Bad rows:")
        for row in bad_rows[:20]:
            print(row)
        raise RuntimeError(f"Found {len(bad_rows)} bad rows in {gt_file}")

    out_manifest.write_text("\n".join(manifest_rows) + "\n")

    print("\nWrote:", out_manifest)
    print("Original rows:", len(lines))
    print("Kept rows:", len(manifest_rows))
    print("Skipped rows:", len(skipped))

    print("\nKept by driver:")
    for driver, count in kept_by_driver.most_common():
        print(f"{driver}: {count}")

    print("\nSkipped by driver:")
    for driver, count in skipped_by_driver.most_common():
        print(f"{driver}: {count}")

    if skipped:
        print("\nFirst skipped examples:")
        for line_idx, driver, reasons in skipped[:20]:
            print(f"line {line_idx}, {driver}")
            for kind, path in reasons:
                print(f"  missing {kind}: {path}")


In [11]:
build_lanedataset_manifest(
    gt_file=TRAIN_GT,
    out_manifest=TRAIN_MANIFEST,
    root=CULANE_ROOT,
    overwrite_binary=False,
    skip_missing=True,
)

build_lanedataset_manifest(
    gt_file=VAL_GT,
    out_manifest=VAL_MANIFEST,
    root=CULANE_ROOT,
    overwrite_binary=False,
    skip_missing=True,
)


Building train.txt: 100%|██████████| 88880/88880 [11:44<00:00, 126.15it/s]



Wrote: /content/CULane/train.txt
Original rows: 88880
Kept rows: 88880
Skipped rows: 0

Kept by driver:
driver_23_30frame: 52857
driver_161_90frame: 18233
driver_182_30frame: 17790

Skipped by driver:


Building val.txt: 100%|██████████| 9675/9675 [01:17<00:00, 125.52it/s]


Wrote: /content/CULane/val.txt
Original rows: 9675
Kept rows: 9675
Skipped rows: 0

Kept by driver:
driver_23_30frame: 9675

Skipped by driver:


TRAINING PART

In [12]:
# =========================
# 1. Imports and config
# =========================
import os
import json
import time
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

import src.config as cfg
from src.data.dataset import LaneDataset
from src.data.transforms import transform_image
from src.models.lanenet import LaneNetResNet34
from src.loss import compute_loss
from src.utils import set_seed, save_checkpoint, load_checkpoint, binary_iou

set_seed(42)

cfg.ROOT_DIR = "/content/CULane"
cfg.BATCH_SIZE = 8
cfg.LEARNING_RATE = 5e-4
cfg.EPOCHS = 50
cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

device = torch.device(cfg.DEVICE)

CKPT_DIR = "/content/drive/MyDrive/Lane_Detection_Project/checkpoints_culane"
os.makedirs(CKPT_DIR, exist_ok=True)

print("device:", device)
print("checkpoint dir:", CKPT_DIR)


device: cuda
checkpoint dir: /content/drive/MyDrive/Lane_Detection_Project/checkpoints_culane


In [13]:
# =========================
# 2. Datasets and loaders
# =========================
train_ds = LaneDataset(
    manifest_path="/content/CULane/train.txt",
    root_dir="/content/CULane",
    transform=transform_image,
    training=True,
)

val_ds = LaneDataset(
    manifest_path="/content/CULane/val.txt",
    root_dir="/content/CULane",
    transform=transform_image,
    training=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

print("train samples:", len(train_ds))
print("val samples:", len(val_ds))
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))


train samples: 88880
val samples: 9675
train batches: 11110
val batches: 1210


In [14]:
# =========================
# 3. Model, optimizer, scheduler
# =========================
model = LaneNetResNet34(
    embedding_dim=4,
    pretrained=True,
).to(device)

backbone_params = list(model.encoder.parameters())
head_params = (
    list(model.binary_decoder.parameters()) +
    list(model.embedding_decoder.parameters())
)

model.encoder.freeze()

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-4},
        {"params": head_params, "lr": 5e-4},
    ],
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.EPOCHS,
)

class_weights = cfg.CLASS_WEIGHTS.to(device)

print(f"params: {sum(p.numel() for p in model.parameters()):,}")
print("class weights:", class_weights)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 159MB/s] 


params: 27,551,014
class weights: tensor([ 1.4540, 20.1856], device='cuda:0')


In [15]:
# =========================
# 4. Train and validation functions
# =========================
def train_one_epoch(model, loader, optimizer, class_weights, device, freeze_encoder_bn=False):
    model.train()

    if freeze_encoder_bn:
        model.encoder.freeze()


    sums = {
        "total": 0.0,
        "binary": 0.0,
        "binary_ce": 0.0,
        "binary_dice": 0.0,
        "disc": 0.0,
        "variance": 0.0,
        "distance": 0.0,
        "reg": 0.0,
    }

    n = 0

    pbar = tqdm(loader, desc="train", leave=False)

    for img, bin_mask, inst_mask in pbar:
        img = img.to(device, non_blocking=True)
        bin_mask = bin_mask.to(device, non_blocking=True)
        inst_mask = inst_mask.to(device, non_blocking=True).long()

        binary_logits, embedding = model(img)

        loss, comps = compute_loss(
            binary_logits,
            embedding,
            bin_mask,
            inst_mask,
            class_weights=class_weights,
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        for k in sums:
            sums[k] += comps[k]

        n += 1
        pbar.set_postfix({k: f"{v / n:.3f}" for k, v in sums.items()})

    return {k: v / n for k, v in sums.items()}


@torch.no_grad()
def validate(model, loader, class_weights, device):
    model.eval()

    sums = {
        "total": 0.0,
        "binary": 0.0,
        "binary_ce": 0.0,
        "binary_dice": 0.0,
        "disc": 0.0,
    }

    iou_sum = 0.0
    n = 0

    for img, bin_mask, inst_mask in tqdm(loader, desc="val", leave=False):
        img = img.to(device, non_blocking=True)
        bin_mask = bin_mask.to(device, non_blocking=True)
        inst_mask = inst_mask.to(device, non_blocking=True).long()

        binary_logits, embedding = model(img)

        _, comps = compute_loss(
            binary_logits,
            embedding,
            bin_mask,
            inst_mask,
            class_weights=class_weights,
        )

        sums["total"] += comps["total"]
        sums["binary"] += comps["binary"]
        sums["binary_ce"] += comps["binary_ce"]
        sums["binary_dice"] += comps["binary_dice"]
        sums["disc"] += comps["disc"]

        iou_sum += binary_iou(binary_logits, bin_mask)
        n += 1

    return {k: v / n for k, v in sums.items()}, iou_sum / n


In [16]:
# =========================
# 5. Early stopping
# =========================
class EarlyStopping:
    def __init__(self, patience=15, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_iou):
        if self.best_score is None:
            self.best_score = val_iou
            return

        if val_iou <= self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_iou
            self.counter = 0


In [17]:
# =========================
# 6. Main training loop
# =========================
EPOCHS = cfg.EPOCHS
WARMUP_EPOCHS = 3

ckpt_path = f"{CKPT_DIR}/last.pt"
best_path = f"{CKPT_DIR}/best.pt"
hist_path = f"{CKPT_DIR}/history.json"

if os.path.exists(ckpt_path):
    start_epoch = load_checkpoint(model, optimizer, ckpt_path, device)
    print(f"Resumed from epoch {start_epoch}")
else:
    start_epoch = 0
    print("Starting from scratch")

if start_epoch > 0:
    for _ in range(start_epoch):
        scheduler.step()


history = []

if os.path.exists(hist_path):
    with open(hist_path, "r") as f:
        history = json.load(f)
    print(f"Loaded history entries: {len(history)}")

best_iou = max((h["val_iou"] for h in history), default=0.0)

early_stopping = EarlyStopping(patience=15, min_delta=0.001)
if history:
    early_stopping.best_score = best_iou

print("starting best_iou:", best_iou)

for epoch in range(start_epoch + 1, EPOCHS + 1):
    if epoch <= WARMUP_EPOCHS:
        model.encoder.freeze()
    else:
        model.encoder.unfreeze()

    t0 = time.time()

    freeze_encoder_bn = epoch <= WARMUP_EPOCHS

    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        class_weights,
        device,
        freeze_encoder_bn=freeze_encoder_bn,
    )

    save_checkpoint(model, optimizer, epoch, ckpt_path)

    torch.cuda.empty_cache()

    val_metrics, val_iou = validate(
        model,
        val_loader,
        class_weights,
        device,
    )

    scheduler.step()
    early_stopping(val_iou)

    elapsed = time.time() - t0

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"train_total={train_metrics['total']:.3f} "
        f"train_bin={train_metrics['binary']:.3f} "
        f"train_disc={train_metrics['disc']:.3f} | "
        f"val_total={val_metrics['total']:.3f} "
        f"val_bin={val_metrics['binary']:.3f} "
        f"val_disc={val_metrics['disc']:.3f} "
        f"val_iou={val_iou:.4f} "
        f"time={elapsed:.0f}s"
    )

    history.append({
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "val_iou": val_iou,
    })

    with open(hist_path, "w") as f:
        json.dump(history, f, indent=2)

    if val_iou > best_iou:
        best_iou = val_iou
        save_checkpoint(model, optimizer, epoch, best_path)
        print(f"New best IoU: {best_iou:.4f}")

    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

print("Training finished.")
print("Best IoU:", best_iou)
print("Best checkpoint:", best_path)


Resumed from epoch 12


/tmp/ipykernel_2596/3607906017.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Loaded history entries: 12
starting best_iou: 0.48734058999140895


[13/50] train_total=0.204 train_bin=0.192 train_disc=0.012 | val_total=0.409 val_bin=0.377 val_disc=0.032 val_iou=0.4790 time=2281s


[14/50] train_total=0.197 train_bin=0.186 train_disc=0.011 | val_total=0.419 val_bin=0.387 val_disc=0.033 val_iou=0.4884 time=2294s
New best IoU: 0.4884


...

[23/50] train_total=0.162 train_bin=0.154 train_disc=0.008 | val_total=0.457 val_bin=0.415 val_disc=0.042 val_iou=0.4856 time=2300s


[24/50] train_total=0.159 train_bin=0.151 train_disc=0.008 | val_total=0.430 val_bin=0.387 val_disc=0.043 val_iou=0.4858 time=2299s


[25/50] train_total=0.155 train_bin=0.147 train_disc=0.008 | val_total=0.455 val_bin=0.409 val_disc=0.045 val_iou=0.4938 time=2299s


train:  70%|██████▉   | 7763/11110 [25:06<10:47,  5.17it/s, total=0.150, binary=0.143, binary_ce=0.085, binary_dice=0.201, disc=0.007, variance=0.005, distance=0.001, reg=2.127]

: 